In [8]:
import pandas as pd 
import numpy as np

In [9]:
rtrain = pd.read_csv("../data/raw/train_ground_truth.csv")
rtest = pd.read_csv("../data/raw/test_ground_truth.csv")

seqa = pd.read_csv("../data/raw/ground_truth_a.csv")
seqb = pd.read_csv("../data/raw/ground_truth_b.csv")
seqc = pd.read_csv("../data/raw/ground_truth_c.csv")

In [10]:
print(seqa.shape, seqb.shape, seqc.shape)

(3600, 11) (3598, 11) (3598, 11)


In [11]:
print(rtrain.columns)
print(seqa.columns)

Index(['IMG_NUM', 'X', 'Y', 'Z', 'ROLL', 'PITCH', 'YAW', 'Q1', 'Q2', 'Q3',
       'W'],
      dtype='str')
Index(['IMG_NUM', 'X', 'Y', 'Z', 'ROLL', 'PITCH', 'YAW', 'Q1', 'Q2', 'Q3',
       'W'],
      dtype='str')


In [12]:
def process_sequence(df, sequence_name):

    df = df.copy()

    # Reset index
    df = df.reset_index(drop=True)

    # Time step from dataset description
    dt = 0.0167

    # Timestamp
    df["timestamp"] = np.arange(len(df)) * dt

    # Identify sequence
    df["sequence"] = sequence_name

    # Rename original pose columns
    df = df.rename(columns={
        "X": "position_x",
        "Y": "position_y",
        "Z": "position_z",

        "ROLL": "roll",
        "PITCH": "pitch",
        "YAW": "yaw",

        "Q1": "quaternion_q1",
        "Q2": "quaternion_q2",
        "Q3": "quaternion_q3",
        "W": "quaternion_w"
    })

    # Derive angular velocity
    df["gyro_x"] = df["roll"].diff() / dt
    df["gyro_y"] = df["pitch"].diff() / dt
    df["gyro_z"] = df["yaw"].diff() / dt

    # First sample has no previous sample
    df["gyro_x"] = df["gyro_x"].fillna(0)
    df["gyro_y"] = df["gyro_y"].fillna(0)
    df["gyro_z"] = df["gyro_z"].fillna(0)

    return df

In [13]:
A_processed = process_sequence(seqa, "A")
B_processed = process_sequence(seqb, "B")
C_processed = process_sequence(seqc, "C")

In [14]:
attitude = seqa[
    ["IMG_NUM", "ROLL", "PITCH", "YAW", "Q1", "Q2", "Q3", "W"]
].copy()

print(attitude.head())

          IMG_NUM       ROLL      PITCH        YAW        Q1        Q2  \
0  img_000000.jpg  45.117479  44.882023  45.166314  0.192013  0.461661   
1  img_000001.jpg  45.234264  44.763820  45.331961  0.192685  0.461381   
2  img_000002.jpg  45.350321  44.645359  45.496938  0.193356  0.461100   
3  img_000003.jpg  45.465648  44.526663  45.661227  0.194026  0.460819   
4  img_000004.jpg  45.580239  44.407755  45.824842  0.194696  0.460536   

         Q3         W  
0  0.192570  0.844344  
1  0.193798  0.844063  
2  0.195025  0.843780  
3  0.196252  0.843496  
4  0.197479  0.843209  


In [15]:
def add_synthetic_telemetry(df, seed=42):

    df = df.copy()

    rng = np.random.default_rng(seed)

    n = len(df)

    # Power
    df["battery_voltage"] = (
        8.1 + rng.normal(0, 0.08, n)
    )

    df["battery_current"] = (
        2.0 + rng.normal(0, 0.2, n)
    )

    df["solar_power"] = (
        20.0 + rng.normal(0, 1.5, n)
    )

    # Thermal
    df["temperature"] = (
        25.0 + rng.normal(0, 1.0, n)
    )

    # Communication
    df["signal_strength"] = (
        -70 + rng.normal(0, 3, n)
    )

    df["packet_loss"] = np.clip(
        rng.normal(0.5, 0.2, n),
        0,
        None
    )

    return df

In [16]:
A_processed = add_synthetic_telemetry(
    A_processed,
    seed=42
)

B_processed = add_synthetic_telemetry(
    B_processed,
    seed=43
)

C_processed = add_synthetic_telemetry(
    C_processed,
    seed=44
)

In [17]:
normal = pd.concat(
    [
        A_processed,
        B_processed,
        C_processed
    ],
    ignore_index=True
)

In [18]:
normal["label"] = 0
normal["fault_type"] = "none"

In [19]:
print(normal["label"].value_counts())

label
0    10796
Name: count, dtype: int64


In [20]:
columns = [
    "timestamp",
    
    "position_x",
    "position_y",
    "position_z",

    "roll",
    "pitch",
    "yaw",

    "quaternion_q1",
    "quaternion_q2",
    "quaternion_q3",
    "quaternion_w",

    "gyro_x",
    "gyro_y",
    "gyro_z",

    "battery_voltage",
    "battery_current",
    "solar_power",

    "temperature",

    "signal_strength",
    "packet_loss",

    "label",
    "fault_type"
]

normal = normal[columns]

In [21]:
normal.to_csv(
    "cubesat_telemetry_normal1.csv",
    index=False
)

## Injecting Faults

In [23]:
import pandas as pd

normal = pd.read_csv("../data/processed/cubesat_telemetry_normal.csv")

print(normal.shape)
print(normal.head())

(10796, 23)
  sequence  timestamp  position_x  position_y  position_z       roll  \
0        A     0.0000        -0.0         0.0         1.0  45.117479   
1        A     0.0167        -0.0         0.0         1.0  45.234264   
2        A     0.0334        -0.0         0.0         1.0  45.350321   
3        A     0.0501        -0.0         0.0         1.0  45.465648   
4        A     0.0668        -0.0         0.0         1.0  45.580239   

       pitch        yaw  quaternion_q1  quaternion_q2  ...    gyro_y  \
0  44.882023  45.166314       0.192013       0.461661  ... -7.078027   
1  44.763820  45.331961       0.192685       0.461381  ... -7.085766   
2  44.645359  45.496938       0.193356       0.461100  ... -7.100504   
3  44.526663  45.661227       0.194026       0.460819  ... -7.113890   
4  44.407755  45.824842       0.194696       0.460536  ... -7.127687   

     gyro_z  battery_voltage  battery_current  solar_power  temperature  \
0  9.918970         8.124377         2.193649  

In [24]:
normal = normal.reset_index(drop=True)

In [25]:
import numpy as np

def inject_thermal_fault(df, start, duration):
    df = df.copy()

    end = start + duration
    indices = df.index[start:end + 1]

    df.loc[indices, "temperature"] = np.linspace(
        25,
        55,
        len(indices)
    )

    df.loc[indices, "label"] = 1
    df.loc[indices, "fault_type"] = "thermal"

    return df

In [26]:
def inject_power_fault(df, start, duration):
    df = df.copy()

    end = start + duration
    indices = df.index[start:end]

    n = len(indices)

    df.loc[indices, "battery_voltage"] = np.linspace(
        8.1,
        6.5,
        n
    )

    df.loc[indices, "battery_current"] = np.linspace(
        2.0,
        3.5,
        n
    )

    df.loc[indices, "solar_power"] = np.linspace(
        20,
        5,
        n
    )

    df.loc[indices, "label"] = 1
    df.loc[indices, "fault_type"] = "power"

    return df

In [27]:
def inject_communication_fault(df, start, duration):
    df = df.copy()

    end = start + duration
    indices = df.index[start:end]

    n = len(indices)

    df.loc[indices, "signal_strength"] = np.linspace(
        -70,
        -95,
        n
    )

    df.loc[indices, "packet_loss"] = np.linspace(
        0.5,
        30,
        n
    )

    df.loc[indices, "label"] = 1
    df.loc[indices, "fault_type"] = "communication"

    return df

In [28]:
def inject_attitude_fault(df, start, duration):
    df = df.copy()

    end = start + duration
    indices = df.index[start:end]

    n = len(indices)

    df.loc[indices, "gyro_x"] = np.linspace(
        0.05,
        2.0,
        n
    )

    df.loc[indices, "gyro_y"] = np.linspace(
        0.03,
        1.5,
        n
    )

    df.loc[indices, "gyro_z"] = np.linspace(
        0.02,
        1.8,
        n
    )

    df.loc[indices, "label"] = 1
    df.loc[indices, "fault_type"] = "attitude"

    return df

In [29]:
test = normal.copy()

test = inject_thermal_fault(
    test,
    start=1000,
    duration=300
)

test = inject_power_fault(
    test,
    start=3000,
    duration=300
)

test = inject_communication_fault(
    test,
    start=5000,
    duration=300
)

test = inject_attitude_fault(
    test,
    start=7000,
    duration=300
)

In [30]:
print(test["fault_type"].value_counts())

fault_type
none             9595
thermal           301
power             300
communication     300
attitude          300
Name: count, dtype: int64


In [31]:
test.to_csv(
    "../data/processed/cubesat_telemetry_test.csv",
    index=False
)

In [32]:
# print(
#     test[
#         test["fault_type"] == "thermal"
#     ][
#         ["temperature", "label", "fault_type"]
#     ].head()
# )

# print(
#     test[
#         test["fault_type"] == "thermal"
#     ][
#         ["temperature", "label", "fault_type"]
#     ].tail()
# )

print(
    test[
        test["fault_type"] == "power"
    ][
        [
            "battery_voltage",
            "battery_current",
            "solar_power",
            "label",
            "fault_type"
        ]
    ].head()
)

      battery_voltage  battery_current  solar_power  label fault_type
3000         8.100000         2.000000    20.000000      1      power
3001         8.094649         2.005017    19.949833      1      power
3002         8.089298         2.010033    19.899666      1      power
3003         8.083946         2.015050    19.849498      1      power
3004         8.078595         2.020067    19.799331      1      power
